In [1]:
!pip install -U ultralytics pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
import random
import sys
import numpy as np
import torch

import yaml
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("yolo26_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file() for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations, load_yaml
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_yolo_final
from scripts.experiment_results import model_training_results_final

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 5
BATCH_SIZE = 8
FRAME_STEP = 15

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [6]:
coco_train = COCO(str(INFO_TRAIN))
coco_val = COCO(str(INFO_VAL))

train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_val, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}
sampled_train_frames = (train_frames.sort_values(["video", "frame_id"]).groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

loading annotations into memory...
Done (t=6.44s)
creating index...
index created!
loading annotations into memory...
Done (t=0.68s)
creating index...
index created!


In [8]:
final_data_dir = PROJECT_ROOT / "yolo_final"
final_data_dir.mkdir(parents=True, exist_ok=True)
class_names = load_yaml(DIRECTORY / "train" / "data.yaml")["names"]
image_lists = {}

for split, frames in [("train", sampled_train_frames), ("val", valid_frames)]:
    image_paths = [(DIRECTORY / split / frame["file_name"]).resolve() 
                   for _, frame in frames.iterrows()]  
    missing = [path for path in image_paths if not path.is_file()]

    if missing:
        raise FileNotFoundError(f"Missing {split} image: {missing[0]}")

    list_path = final_data_dir / f"{split}.txt"
    list_path.write_text("\n".join(str(path) for path in image_paths) + "\n", encoding="utf-8",)
    image_lists[split] = str(list_path.resolve())

final_yaml_path = final_data_dir / "data.yaml"
config = {"path": str(DIRECTORY.resolve()), "train": image_lists["train"],
    "val": image_lists["val"], "names": class_names
}

final_yaml_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

237

In [9]:
names = (dict(enumerate(class_names)) if isinstance(class_names, list) else class_names)
category_by_name = {category["name"]: category["id"] for category in coco_val.cats.values()}
class_to_category = {int(index): category_by_name[name] for index, name in names.items()}

valid_dataset = DetectionDataset(frames=valid_frames, directory=DIRECTORY, 
                                 annotations_by_image=valid_annotations_by_image, 
                                 empty_annotations=valid_boxes.iloc[:0].copy(), validation=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0,
                        collate_fn=collate_data)

In [10]:
output_root = PROJECT_ROOT / "outputs" / f"yolo_final"

train_kwargs = {"data_yaml": final_yaml_path, "valid_loader": valid_loader,
    "coco_gt": coco_val, "class_to_category": class_to_category,
    "device": (str(device.index if device.index is not None else 0) if device.type == "cuda" else device.type), "fold": "final",
    "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "lr": 1e-4, "weight_decay": 1e-4,
    "optimizer": "Adam", "seed": SEED
}

In [11]:
tuned_history, tuned_results = run_yolo_final(name="tuned", imgsz=1024,
    box_weight=10.0, train_kwargs=train_kwargs,
    output_root=output_root, seed=SEED
)

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_final/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_final, nbs=64, nms=None, opset=None, optimize

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/5       4.3G      2.736      1.922   0.003247        340       1024: 100% ━━━━━━━━━━━━ 206/206 4.6it/s 45.1s0.2ss
Loading and preparing results...
DONE (t=0.34s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=91.40s).
Accumulating evaluation results...
DONE (t=3.37s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.103
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.193
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.094
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.038
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.083
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.241
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.054
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.149
 Average Recall     (AR

In [ ]:
tuned_results_wrap = model_training_results_final(output_root/"tuned"/"fold_final")

Training history


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.284084,0.000033,0.086469,0.155837,0.031408
1,2,4.661068,0.000053,0.102943,0.192659,0.037619
2,3,4.511594,0.000060,0.119186,0.220305,0.040655
3,4,4.402048,0.000041,0.124889,0.230322,0.042216
4,5,4.360668,0.000021,0.125518,0.230968,0.043364



Best validation results — epoch 5


,mAP,AP50,AP_small
4,0.125518,0.230968,0.043364
